In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Cleaned data load karo
transactions_clean = pd.read_csv(r'C:\Users\HP\Desktop\Project 2 - Cohort - Retention - CLTV\Data\processed\cleaned_transactions.csv')
first_purchase = pd.read_csv(r'C:\Users\HP\Desktop\Project 2 - Cohort - Retention - CLTV\Data\processed\first_purchase.csv')

# Date columns fix karo
transactions_clean['purchase_date'] = pd.to_datetime(transactions_clean['purchase_date'])
transactions_clean['cohort_month'] = pd.to_datetime(transactions_clean['cohort_month'].astype(str))
first_purchase['first_purchase_date'] = pd.to_datetime(first_purchase['first_purchase_date'])

print("Data Loaded ✅")
print("Shape:", transactions_clean.shape)

Data Loaded ✅
Shape: (2211, 11)


In [19]:
# Har transaction ka purchase month nikalo
transactions_clean['purchase_month'] = transactions_clean['purchase_date'].dt.to_period('M')
transactions_clean['cohort_month'] = transactions_clean['cohort_month'].dt.to_period('M')

# Groupby check
cohort_data = transactions_clean.groupby(['cohort_month', 'purchase_month'])['user_id'].nunique().reset_index()
cohort_data.columns = ['cohort_month', 'purchase_month', 'users']

print("=== COHORT DATA SAMPLE ===")
print(cohort_data.head(10))

=== COHORT DATA SAMPLE ===
  cohort_month purchase_month  users
0      2023-01        2023-01    107
1      2023-01        2023-02     15
2      2023-01        2023-03     14
3      2023-01        2023-04      4
4      2023-01        2023-05      2
5      2023-02        2023-02    104
6      2023-02        2023-03     17
7      2023-02        2023-04     18
8      2023-02        2023-05      7
9      2023-02        2023-06      3


In [21]:
# Cell 3 - Months Difference (Issue #8)
transactions_clean['cohort_month'] = pd.PeriodIndex(transactions_clean['cohort_month'])
transactions_clean['purchase_month'] = pd.PeriodIndex(transactions_clean['purchase_month'])

# Months difference calculate karo
transactions_clean['months_since_acquisition'] = (
    transactions_clean['purchase_month'] - transactions_clean['cohort_month']
).apply(lambda x: x.n)

print("=== MONTHS SINCE ACQUISITION ===")
print(transactions_clean[['user_id', 'cohort_month', 'purchase_month', 'months_since_acquisition']].head(10))
print("\nMax months:", transactions_clean['months_since_acquisition'].max())
print("Min months:", transactions_clean['months_since_acquisition'].min())

=== MONTHS SINCE ACQUISITION ===
     user_id cohort_month purchase_month  months_since_acquisition
0  USR_00001      2024-01        2024-01                         0
1  USR_00002      2023-10        2023-10                         0
2  USR_00003      2023-06        2023-06                         0
3  USR_00004      2023-01        2023-01                         0
4  USR_00005      2023-07        2023-07                         0
5  USR_00005      2023-07        2023-07                         0
6  USR_00006      2023-07        2023-07                         0
7  USR_00006      2023-07        2023-08                         1
8  USR_00008      2023-09        2023-09                         0
9  USR_00010      2023-01        2023-01                         0

Max months: 7
Min months: 0


In [23]:
# Cell 4 - Cohort Pivot Table (Issue #9)
cohort_pivot = transactions_clean.groupby(
    ['cohort_month', 'months_since_acquisition']
)['user_id'].nunique().reset_index()

cohort_pivot = cohort_pivot.pivot_table(
    index='cohort_month',
    columns='months_since_acquisition',
    values='user_id'
)

print("=== COHORT PIVOT TABLE ===")
print(cohort_pivot.head())
print("\nShape:", cohort_pivot.shape)

=== COHORT PIVOT TABLE ===
months_since_acquisition      0     1     2    3    4    5   6   7
cohort_month                                                      
2023-01                   107.0  15.0  14.0  4.0  2.0  NaN NaN NaN
2023-02                   104.0  17.0  18.0  7.0  3.0  1.0 NaN NaN
2023-03                   138.0  28.0  15.0  4.0  NaN  NaN NaN NaN
2023-04                   124.0  25.0  21.0  5.0  2.0  NaN NaN NaN
2023-05                   114.0  15.0  15.0  4.0  1.0  NaN NaN NaN

Shape: (16, 8)


In [25]:
# Cell 5 - Retention Percentages (Issue #10)
cohort_size = cohort_pivot.iloc[:, 0]

retention_matrix = cohort_pivot.divide(cohort_size, axis=0) * 100
retention_matrix = retention_matrix.round(2)

print("=== RETENTION MATRIX (%) ===")
print(retention_matrix.head())
print("\nShape:", retention_matrix.shape)

=== RETENTION MATRIX (%) ===
months_since_acquisition      0      1      2     3     4     5   6   7
cohort_month                                                           
2023-01                   100.0  14.02  13.08  3.74  1.87   NaN NaN NaN
2023-02                   100.0  16.35  17.31  6.73  2.88  0.96 NaN NaN
2023-03                   100.0  20.29  10.87  2.90   NaN   NaN NaN NaN
2023-04                   100.0  20.16  16.94  4.03  1.61   NaN NaN NaN
2023-05                   100.0  13.16  13.16  3.51  0.88   NaN NaN NaN

Shape: (16, 8)
